In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# ====================================================
# 1. KHỞI TẠO SPARK KẾT NỐI VỚI CLUSTER
# ====================================================
print("🚀 Đang khởi động Spark kết nối Cluster...")
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Cluster_Process_Articles_Multimodal") \
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.memory", "3g") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Đã kết nối Spark thành công!")

🚀 Đang khởi động Spark kết nối Cluster...


26/03/23 15:57:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Đã kết nối Spark thành công!


In [2]:
# ====================================================
# 2. KHAI BÁO ĐƯỜNG DẪN HDFS & ĐỌC DỮ LIỆU
# ====================================================
INPUT_FILE = "hdfs://namenode:9000/data/raw/articles.csv" 

print(f"1️⃣ Đang đọc bảng Sản Phẩm từ: {INPUT_FILE}")
df_articles = spark.read.csv(INPUT_FILE, header=True, inferSchema=True)


print("✅ Đọc file thành công!")

1️⃣ Đang đọc bảng Sản Phẩm từ: hdfs://namenode:9000/data/raw/articles.csv


✅ Đọc file thành công!


In [3]:
# ---------------------------------------------------------
# BƯỚC 1 & 2: XỬ LÝ ID VÀ LỌC SẢN PHẨM HỢP LỆ
# ---------------------------------------------------------
print("⚙️ Đang chuẩn hóa ID và Join dữ liệu...")

# Thêm số 0 vào trước ID để đủ 10 ký tự
df_articles = df_articles.withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))


print("✅ Đã lọc xong những sản phẩm hợp lệ!")
# Nếu muốn xem còn lại bao nhiêu sản phẩm, bỏ comment dòng dưới (chạy sẽ hơi lâu 1 chút):
# print("Số sản phẩm sau khi lọc:", df_joined.count())
df_articles.count()

⚙️ Đang chuẩn hóa ID và Join dữ liệu...
✅ Đã lọc xong những sản phẩm hợp lệ!


105542

In [4]:
# ---------------------------------------------------------
# BƯỚC 3: XỬ LÝ MISSING VALUES (Giá trị thiếu)
# ---------------------------------------------------------
print("3️⃣ Đang điền các giá trị trống (NULL) thành 'Unknown'...")

cols_to_fill = [
    "prod_name", "product_type_name", "product_group_name", 
    "colour_group_name", "department_name", "detail_desc"
]

df_cleaned = df_joined.fillna({c: "Unknown" for c in cols_to_fill})
print("✅ Đã xử lý xong Missing Values!")

3️⃣ Đang điền các giá trị trống (NULL) thành 'Unknown'...


NameError: name 'df_joined' is not defined

In [ ]:
# ---------------------------------------------------------
# BƯỚC 4: LỌC CỘT VÀ LƯU TRỮ TRÊN HDFS
# ---------------------------------------------------------
final_cols = [
    "article_id", "prod_name", "product_type_name", 
    "product_group_name", "colour_group_name", "detail_desc" 
]

df_final = df_cleaned.select(final_cols)

# Hiển thị 5 dòng để nghiệm thu
print("👀 5 dòng dữ liệu chuẩn bị lưu:")
df_final.show(5, truncate=False)

# Lưu ra Parquet trên Server
print("4️⃣ Đang ghi ra file Parquet lên HDFS...")
df_final.write.mode("overwrite").parquet(OUTPUT_FILE)
print(f"🎉 Hoàn tất! File Articles chuẩn đã lưu tại: {OUTPUT_FILE}")

In [ ]:
# ====================================================
# 3. ĐÓNG KẾT NỐI SPARK 
# ====================================================
spark.stop()
print("🛑 Đã ngắt kết nối Spark, giải phóng tài nguyên cho hệ thống.")

26/03/23 16:04:50 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/03/23 16:04:50 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:873)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:154)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:262)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:169)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:100)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$spark$rpc$netty$Mess